# 04 — A graph neural network baseline

Notebook 02 turned a catalog into 32 numbers and fit a Ridge regression to them. That works,
and it is invariant to every symmetry condition, but look at what it throws away: after
histogramming, a catalog where the galaxies sit in a tight filament and one where they are
scattered at random can have **the same feature vector**. Only the four neighbour counts see
geometry at all, and they see it as four scalars.

This notebook keeps the geometry. Each catalog becomes a **graph**: galaxies are nodes,
nearby pairs are edges, and a message-passing network reads the whole thing.

The model is the one from **de Santi et al. 2023** ([arXiv:2302.14101](https://arxiv.org/abs/2302.14101)),
which is the published state of the art for exactly this task — inferring cosmology from a
CAMELS galaxy catalog. We ship it as the reference submission, so you have a real floor to
beat rather than a toy.

By the end you will have:

1. built a graph from a catalog and looked at it,
2. seen the three edge features and *checked* that they are symmetry-invariant,
3. found a symmetry break hiding in the published recipe,
4. trained the network and scored it,
5. measured what the symmetry conditions actually cost the trained model.

**This notebook needs a GPU.** On one A100 the whole thing runs in about ten minutes, most
of it reading catalogs off disk. Everything before the training cell runs fine on a CPU.

A reminder of the vocabulary from notebook 03, since the tables below use it. The test
conditions come in three kinds. **Symmetries** (`translate`, `rotate90`, `permute`, marked
tier `S`) do not change the physics at all, so a correctly built model must score the same
before and after — if it does not, that is an architecture bug, not a robustness limit.
**Corruptions** (`pos_noise`, `vel_noise`, `mass_noise`, tier `C`) genuinely destroy
information, so losing accuracy is expected and the question is how much. **A different
simulation code** is the third kind, and it is the one the event is really about.

In [ ]:
# Setup. On Colab this installs the toolkit, mounts the data bucket and points the
# environment variables at it. Anywhere else -- a cluster with the data already on disk --
# it does nothing, which is why there is one set of notebooks rather than two.
import sys

if "google.colab" in sys.modules:
    # --force-reinstall, every time, on purpose. Installing only when the package is
    # missing means anyone who ran a notebook once keeps a stale copy forever, and during
    # an event where fixes are being pushed that is exactly backwards. --no-deps keeps it
    # to a few seconds: everything it depends on is already in the runtime.
    %pip install -q --upgrade --force-reinstall --no-deps git+https://github.com/xwzhang98/kaai-robust-inference-hackathon-2026
    # Drop anything already imported from the old copy, so this works without a restart.
    for _name in [m for m in sys.modules if m.startswith("kaai_hackathon")]:
        del sys.modules[_name]

from kaai_hackathon.colab_setup import setup

setup()

## Setup

Same data convention as the earlier notebooks: `CAMELS_HACKATHON_DATA` points at a directory
laid out as `<suite>/LH_<n>/groups_090.hdf5`.

In [ ]:
%matplotlib inline
import os
import time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

from kaai_hackathon import PUBLIC_SUITES
from kaai_hackathon.catalog_io import read_catalog
from kaai_hackathon.splits import example_sims, load_labels, local_split

DATA_ROOT = Path(os.environ["CAMELS_HACKATHON_DATA"])
PARAMS_ROOT = Path(os.environ.get("CAMELS_HACKATHON_PARAMS", DATA_ROOT))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# The shipped baseline's settings, used everywhere below so that what this notebook builds
# and what baseline/gnn/model.pt is are the same thing. Section 8 reads them back off the
# checkpoint so you can check that claim rather than take it.
USE_MSTAR = True                          # node = [v_z, log10(1 + M_star)]
NODE_DIM = 2 if USE_MSTAR else 1
R_LINK_TUNED = 0.038358368230528794       # 0.96 cMpc/h, not the paper's 1.25


def catalog_path(suite, sim_id):
    return DATA_ROOT / suite / f"LH_{sim_id}" / "groups_090.hdf5"


def load(suite, sim_id):
    return read_catalog(catalog_path(suite, sim_id), group_fields=[],
                        subhalo_fields=["SubhaloPos", "SubhaloVel", "SubhaloMassType"])

# A simulation that is certain to exist. The public ids are a pinned random 900
# out of 1000, not 0..899, so `LH_2` may simply not be in the data you were given.
EXAMPLE = example_sims("IllustrisTNG")[0]
EXAMPLE2 = example_sims("IllustrisTNG", 2)[1]



print("device:", DEVICE)
if DEVICE.type == "cpu":
    print("WARNING: no GPU visible. Everything runs, but the training cell will be slow.")

## 1. From a set of galaxies to a graph

Three decisions define the graph, and all three are yours to change:

| decision | de Santi's choice | why |
|---|---|---|
| which galaxies are nodes | $M_\star > 1.95\times10^{8}\,M_\odot/h$ at test time, **randomized** over $1.3\text{–}2.6\times10^{8}$ while training | see below; the randomization is not cosmetic |
| when two nodes are connected | separation $< 1.25$ cMpc$/h$ | their tuned linking length |
| what a node carries | $\operatorname{sign}(v_z)\log_{10}(1+\lvert v_z\rvert)$ | the line-of-sight velocity, and nothing else |

**Where are the positions?** Nowhere on the node, and that is the design rather than an
oversight. Positions decide *which pairs are connected*, and they set the distance and the
two angles carried on each edge. They never enter as a coordinate. Put an absolute position
on a node and the model stops being translation- and rotation-invariant, which is exactly
what the symmetry conditions test.

**Why randomize the mass cut?** The faintest galaxy a simulation can resolve is set by its
particle mass, and in CAMELS the particle mass is a direct function of $\Omega_m$. Train at
one fixed cut and the model can read $\Omega_m$ off *where the mass function stops* instead
of off the physics. Redrawing the cut per catalog closes that shortcut, and as a side effect
it makes the model tolerate a catalog selected differently from the ones it trained on --
which is the same thing this event calls robustness.

The linking length is the other one worth thinking about. Too small and the graph falls apart
into isolated points, and you are back to a bag of galaxies. Too large and every galaxy
connects to every other, the edges stop saying *which* galaxies are close, and the cost grows
as $N^2$.

`catalog_to_graph` does all of it. Note that it normalizes positions to $[0,1)$ -- the whole
graph module works in box units, so the linking length is $1.25/25 = 0.05$.

In [ ]:
from kaai_hackathon.graph import R_LINK, catalog_to_graph

cat = load("IllustrisTNG", EXAMPLE)
graph = catalog_to_graph(cat, r_link=R_LINK_TUNED, use_mstar=USE_MSTAR)

print(f"r_link = {R_LINK_TUNED:.4f} box units = {R_LINK_TUNED * 25:.2f} cMpc/h "
      f"(the paper's untuned value is {R_LINK} = {R_LINK * 25:.2f})")
print()
print(f"x           {str(graph['x'].shape):>12}   node features, [v_z, log10(1 + M_star)]")
print(f"edge_index  {str(graph['edge_index'].shape):>12}   [src; dst], both directions per pair")
print(f"edge_attr   {str(graph['edge_attr'].shape):>12}   three features per edge")
print(f"u           {str(graph['u'].shape):>12}   one global feature, log10(N_galaxies)")
print()
degree = np.bincount(graph["edge_index"][1], minlength=len(graph["x"]))
print(f"{len(graph['x'])} galaxies, {graph['edge_index'].shape[1] // 2} pairs")
print(f"degree: mean {degree.mean():.1f}, median {np.median(degree):.0f}, "
      f"max {degree.max()}, isolated {int((degree == 0).sum())}")

A good fraction of galaxies are isolated at this linking length, and that is fine — the
readout pools over all nodes, so an isolated galaxy still contributes its mass. What it does
not contribute is any information about *where* it is, which is one honest limitation of the
architecture.

Let's look at the graph. Drawing all of it is a hairball, so take a thin slab.

In [ ]:
pos = np.mod(cat.subhalo["SubhaloPos"] / cat.box_size, 1.0)
mstar = cat.subhalo["SubhaloMassType"][:, 4]
selected = pos[mstar > 1.3e-2]

SLAB = 0.12
in_slab = selected[:, 2] < SLAB
src, dst = graph["edge_index"]
both_in = in_slab[src] & in_slab[dst] & (src < dst)

fig, ax = plt.subplots(figsize=(7, 7))
for a, b in zip(src[both_in], dst[both_in]):
    p, q = selected[a], selected[b]
    if np.abs(p - q).max() > 0.5:      # a pair joined across the periodic boundary
        continue
    ax.plot([p[0], q[0]], [p[1], q[1]], lw=0.7, color="C0", alpha=0.55, zorder=1)
ax.scatter(selected[in_slab, 0], selected[in_slab, 1], s=9, color="k", zorder=2)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
ax.set_xlabel("x / box"); ax.set_ylabel("y / box")
ax.set_title(f"IllustrisTNG LH_0, slab z < {SLAB * 25:.0f} cMpc/h "
             f"({int(both_in.sum())} edges drawn)")
plt.tight_layout()

The edges trace the filaments. That is the point: the graph is a discrete model of the cosmic
web, and how strongly it clumps is what $\sigma_8$ controls.

(Pairs joined across the periodic boundary are skipped in the drawing only — they are real
edges in the graph. `periodic_radius_graph` builds them with `scipy.spatial.KDTree(...,
boxsize=...)`, which wraps natively.)

How sensitive is the graph to the linking length?

In [ ]:
radii = np.array([0.01, 0.02, 0.03, 0.05, 0.08, 0.12, 0.2])
rows = []
for suite in PUBLIC_SUITES:
    c = load(suite, example_sims(suite)[0])
    rows.append([catalog_to_graph(c, r_link=float(r), use_mstar=USE_MSTAR)
                 ["edge_index"].shape[1] // 2 for r in radii])

fig, ax = plt.subplots(figsize=(6.4, 4.2))
for suite, counts in zip(PUBLIC_SUITES, rows):
    ax.plot(radii * 25, counts, marker="o", ms=4, label=suite)
ax.axvline(R_LINK_TUNED * 25, ls="--", c="0.4", lw=1)
ax.text(R_LINK_TUNED * 25 * 1.06, ax.get_ylim()[1] * 0.5, "our r_link", color="0.4")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("linking length [cMpc/h]"); ax.set_ylabel("pairs in LH_0")
ax.legend(); plt.tight_layout()

Roughly a cubic law, as you would expect from a volume — and the three suites are offset from
each other because they make different numbers of galaxies. **That offset is a distribution
shift built into the graph itself.** A model trained on one suite meets a systematically
denser or sparser graph on another, before anything about the physics differs.

## 2. The edge features, and why they are invariant

Each edge carries three numbers (de Santi Eqs. 3–7). Write $c$ for a reference point in the
catalog and $\delta_i = r_i - c$:

$$e_{ij} = \Big[\; \frac{|d_{ij}|}{r_\mathrm{link}}, \quad
\hat\delta_i \cdot \hat\delta_j, \quad
\hat\delta_i \cdot \hat d_{ij} \;\Big],
\qquad d_{ij} = r_i - r_j$$

All three differences use the minimum-image convention. The first is a distance; the other two
are cosines of angles. **No absolute position and no absolute direction appears anywhere** —
which is why the model is rotation-invariant by construction rather than by augmentation.

That is the argument. Here is the measurement.

In [ ]:
from kaai_hackathon.graph import edge_features, periodic_radius_graph
from kaai_hackathon.shifts import cubic_rotations


def keyed_features(positions, centroid="circular", r_link=R_LINK):
    # Edge features keyed by (src, dst), so two graphs can be compared pair by pair.
    src, dst = periodic_radius_graph(positions, r_link=r_link)
    e = edge_features(positions, src, dst, r_link=r_link, centroid=centroid)
    return {(int(a), int(b)): row for a, b, row in zip(src, dst, e)}


def compare(a, b):
    # Largest feature difference over the edges the two graphs share.
    shared = set(a) & set(b)
    worst = max(float(np.abs(a[k] - b[k]).max()) for k in shared)
    return worst, len(shared), max(len(a), len(b))


rng = np.random.default_rng(0)
base = np.mod(selected, 1.0).astype(np.float32)
centre = np.full(3, 0.5)
rotation = cubic_rotations()[7]

variants = {
    "translate": np.mod(base + np.float32(0.137), 1.0).astype(np.float32),
    "rotate90": np.mod((base - centre) @ rotation.T + centre, 1.0).astype(np.float32),
    "permute": base[rng.permutation(len(base))],
}

reference = keyed_features(base)
print(f"{'shift':12s} {'max |delta e|':>14s}   shared edges")
for name, moved in variants.items():
    if name == "permute":
        # row order changes the *labels*, so compare the multiset of feature rows instead
        a = np.sort(np.stack(list(reference.values())), axis=0)
        b = np.sort(np.stack(list(keyed_features(moved).values())), axis=0)
        print(f"{name:12s} {np.abs(a - b).max():14.2e}   (compared as a multiset)")
    else:
        worst, shared, total = compare(reference, keyed_features(moved))
        print(f"{name:12s} {worst:14.2e}   {shared} / {total}")

Three symmetries, three answers at the level of floating-point noise. The differences are
not exactly zero because wrapping the box in `float32` moves positions by about
$10^{-3}$ ckpc$/h$; the shared-edge count is printed for the same reason, since a pair sitting
within that distance of the linking length can in principle flip in or out. Both are rounding,
not symmetry breaks. What they do mean is that a graph model's score under the symmetry
conditions will be *very close to* unchanged rather than bit-identical, the way the
histogram model's was.

### The one that is not invariant

Everything above used `centroid="circular"`. The published recipe uses the arithmetic mean of
the wrapped coordinates as the reference point $c$, and an arithmetic mean is **not a periodic
quantity**: shift every galaxy by the same vector, wrap, and the mean does not shift with them.

Watch what that does.

In [ ]:
print(f"{'centroid':12s} {'translate':>12s} {'rotate90':>12s}")
for mode in ("mean", "circular"):
    row = []
    for name in ("translate", "rotate90"):
        worst, _, _ = compare(keyed_features(base, centroid=mode),
                              keyed_features(variants[name], centroid=mode))
        row.append(worst)
    print(f"{mode:12s} {row[0]:12.2e} {row[1]:12.2e}")

print("\nThe features are cosines and a scaled distance, so O(1) is total corruption,")
print("not a small perturbation.")

So the published edge features fail one of the three symmetry conditions — not because the idea
is wrong, but because in the setting it was designed for nobody ever translates the box, so
the bug is invisible. Here it is a scored condition.

The fix is one line: use the **circular mean**, which treats each axis as an angle,

$$c_k = \frac{1}{2\pi}\,\operatorname{atan2}\big(\langle \sin 2\pi x_k \rangle,\;
\langle \cos 2\pi x_k \rangle\big),$$

and therefore rotates with the box. It is equivariant under the cubic rotations too, so
nothing is given up. `catalog_to_graph` defaults to it; `centroid="mean"` reproduces the paper.

### How much does that actually cost?

A feature moving by 2.0 looks alarming. Before believing it matters, measure it on the thing
that gets scored — the trained model's $R^2$ — because a change in an input is not the same
as a change in an answer.

Trained on all three suites with the tuned configuration below, one seed:

| centroid | clean $\Omega_m$ | clean $\sigma_8$ | change under `translate` |
|---|---|---|---|
| `"mean"` (as published) | 0.896 | 0.265 | −0.002 / −0.006 |
| `"circular"` (the fix) | 0.890 | 0.242 | 0.000 / 0.000 |

So the honest answer is: **the break is real and the cost is small.** The reason is the
readout — it pools over every edge in the catalog, and moving the reference point leaves the
*distribution* of those angles nearly unchanged even though every individual value moves. The
pooled representation barely notices.

That is worth taking in both directions. A large change in a feature does not automatically
mean a large change in a score, so measure rather than assume — including when the thing you
are assuming is your own alarming-looking finding. And a small cost is still a cost you did
not have to pay: the circular centroid removes it exactly, for one line, with no training and
no data.

### The node feature is where invariance stops

The edge features survive all three symmetries. The published *node* feature survives only
two, and this one is not a bug — it is a modelling choice with a consequence.

$v_z$ is **one projected component** of the velocity. Turn the box by a right angle that
moves the $z$ axis and every node's feature becomes a different component of the same vector.
The universe is isotropic, so the *distribution* of $v_z$ is unchanged; this particular
catalog's numbers are not, and the model's answer moves with them.

In [ ]:
from kaai_hackathon.graph import node_features

vel = cat.subhalo["SubhaloVel"][:400].astype(np.float64)
turn_z_to_y = np.array([[1, 0, 0], [0, 0, -1], [0, 1, 0]], dtype=np.float64)
turned = vel @ turn_z_to_y.T

for mode in ("vz", "speed"):
    before = node_features(vel, velocity=mode)
    after = node_features(turned, velocity=mode)
    print(f"velocity={mode:6s}  largest change under a 90-degree turn = "
          f"{np.abs(before - after).max():.3e}")

This one costs more than the centroid did. Here is every node feature we trained: same
configuration, one seed each, 900 simulations per suite, scored on the held-out 100.

| node feature | clean $\Omega_m$ | clean $\sigma_8$ | change under `rotate90` |
|---|---|---|---|
| $v_z$ | 0.890 | 0.242 | −0.008 / **−0.028** |
| $v_z$, with the published centroid too | 0.896 | 0.265 | −0.002 / **−0.050** |
| $\log_{10}(1+\lvert v\rvert)$ (`velocity="speed"`) | **0.914** | 0.254 | **0.000 / 0.000** |
| $\log_{10}(1+M_\star)$ only, no velocity | 0.668 | 0.378 | 0.000 / 0.000 |
| **$v_z$ and $\log_{10}(1+M_\star)$ — the shipped baseline** | **0.903** | **0.403** | −0.014 / −0.002 |

Two separate things are visible there, and both are worth having.

**The rotation break is real.** $\sigma_8$ loses 0.03 to 0.05 under a right-angle turn on the
velocity-only rows and exactly nothing on the two rows whose node feature is rotation
invariant. Adding the mass column dilutes it to −0.002, because a mass does not care which
way the box is facing.

**Velocity and mass carry different parameters.** Velocity alone gets $\Omega_m$ to 0.89 and
leaves $\sigma_8$ at 0.24. Mass alone does the reverse: 0.67 and 0.38. Give the node both and
you get 0.90 **and** 0.40 — better than either alone, on both targets. That is why the shipped
baseline uses both, and it is a fair example of the kind of thing worth trying here: the
published fiducial model was tuned for $\Omega_m$ alone, and this event scores two parameters.

Which velocity you want is still a real decision rather than an obvious one. The line-of-sight
projection is the right stand-in for an observable and carries redshift-space information the
speed throws away. The point of measuring is to choose deliberately rather than find out on
the leaderboard.

## 3. The network

MetaLayer blocks (de Santi Eqs. 9–12). One block updates the edges, then the nodes:

$$e_{ij}' = E\big([\,n_i,\; n_j,\; e_{ij}\,]\big), \qquad
n_i' = N\Big(\big[\,n_i,\; \textstyle\sum_j e_{ij}',\; \max_j e_{ij}',\;
\operatorname{mean}_j e_{ij}',\; g\,\big]\Big)$$

and after the last block a readout pools over the whole catalog:

$$y = F\Big(\big[\,\textstyle\sum_i n_i,\; \operatorname{mean}_i n_i,\; \max_i n_i,\; g\,\big]\Big)$$

$E$, $N$ and $F$ are small MLPs. Two details are worth pausing on.

**The pooling is what makes it a set model.** Sum, mean and max do not care about row order,
so `row_permutation` cannot touch the output. The variable catalog size is handled for free
by the same mechanism.

**The global feature $g = \log_{10} N_\mathrm{gal}$ is fed in deliberately** (paper footnote 3),
into every node update and into the readout. So the number of galaxies is an explicit input.
Worth knowing when you interpret the scores: some of what the model achieves is available from
abundance alone, and notebook 02's count feature already gets part of it.

The output is $2 \times$ the number of parameters: a posterior mean $\mu$ and a width $\sigma$
for each.

In [ ]:
from kaai_hackathon.gnn import DeSantiGNN, collate, moment_loss
from kaai_hackathon.graph import sample_mstar_threshold

model = DeSantiGNN(node_features=NODE_DIM, edge_features=3, n_global=1,
                   hidden=64, n_layers=1, n_params=2)
print(model.blocks[0])
print(f"\ntrainable parameters: {sum(p.numel() for p in model.parameters()):,}"
      " -- the same count as the shipped checkpoint, which is one way to tell you are"
      " building the same model")

Batching graphs of different sizes is done by gluing them into one big disconnected graph and
carrying a `batch` vector saying which catalog each node came from. `collate` does it; that
vector is what turns the readout's pooling into a per-catalog pooling.

In [ ]:
pair = collate([catalog_to_graph(load("IllustrisTNG", EXAMPLE), r_link=R_LINK_TUNED,
                                 use_mstar=USE_MSTAR),
                catalog_to_graph(load("IllustrisTNG", EXAMPLE2), r_link=R_LINK_TUNED,
                                 use_mstar=USE_MSTAR)])
for key in ("x", "edge_index", "edge_attr", "u", "batch"):
    print(f"{key:11s} {tuple(pair[key].shape)}")
print("\nbatch vector counts:", torch.bincount(pair["batch"]).tolist())

## 4. The loss

Ordinary mean-squared error gives you a point estimate and nothing else. The Jeffrey–Wandelt
moment loss (de Santi Eq. 20) asks for a mean **and** a width:

$$\mathcal{L} = \log\Big(\big\langle\textstyle\sum_i (\theta_i-\mu_i)^2\big\rangle\Big)
+ \log\Big(\big\langle\textstyle\sum_i \big[(\theta_i-\mu_i)^2-\sigma_i^2\big]^2\big\rangle\Big)$$

The first term fits $\mu$. The second is minimized when $\sigma^2$ equals the *actual* squared
error, so $\sigma$ ends up being a calibrated uncertainty rather than a free parameter. The two
logarithms put the terms on the same scale so the first cannot swamp the second.

We only score $\mu$ — but a model that also reports how sure it is tells you much more about
where it is failing, and the OOD condition is exactly where you want that.

In [ ]:
theta = torch.tensor([[0.3], [0.5]])
mu = torch.tensor([[0.4], [0.4]])                     # squared error is 0.01 in both rows
for label, sigma in (("under-confident sigma=0.5", 0.5),
                     ("calibrated    sigma=0.1", 0.1),
                     ("over-confident  sigma=0.01", 0.01)):
    prediction = torch.cat([mu, torch.full_like(mu, sigma)], dim=-1)
    print(f"{label:28s} loss = {moment_loss(prediction, theta)[0].item():+.3f}")

## 5. Train it

Below is a **scaled-down** run so this notebook finishes: fewer simulations, fewer epochs.
The numbers you get here will be worse than the shipped baseline; section 8 points at the
full-scale scores.

Scale it down much further and the model stops learning altogether — it settles on predicting
the same numbers for every catalog, which drives the loss to about $-6.2$ and every $R^2$ to
about $0$. That is worth seeing at least once, because it is the failure you will hit if you
develop on a small subset: **a flat loss curve and identical scores under every condition are
the signature of a model that has given up, not of a robust one.**

Two choices in this cell are not just bookkeeping:

- **Targets are mapped onto $[0,1]$** using the known prior ranges. $\Omega_m \in [0.1, 0.5]$
  and $\sigma_8 \in [0.6, 1.0]$, and the loss sums over parameters, so without this the fit is
  quietly weighted towards whichever has the larger raw spread.
- **The checkpoint is selected on validation loss**, never training loss. This is the whole
  reason the validation split exists.

In [ ]:
# The shipped baseline uses all 900 simulations per suite, 5 augmentations and 300 epochs.
# Everything else matches it, including the hyperparameters, which come from an Optuna search
# in the reference port of de Santi's code rather than from taste.
N_TRAIN_PER_SUITE = 300
N_AUG = 3                   # copies per simulation, each with its own random mass cut
EPOCHS = 150
BATCH_SIZE = 25
VAL_FRACTION = 0.15

N_LAYERS = 1                # the tuned depth. Three message-passing blocks overfits hard:
                            # the training loss keeps falling while validation turns around
                            # near epoch 20, and the selected checkpoint is no better.
HIDDEN = 64
BASE_LR, MAX_LR = 1.066631364804843e-05, 1e-3
WEIGHT_DECAY = 1.0730720816241572e-07
# R_LINK_TUNED, USE_MSTAR and NODE_DIM were set in the setup cell.

PRIOR_LO = np.array([0.1, 0.6])          # Omega_m, sigma_8
PRIOR_HI = np.array([0.5, 1.0])

from kaai_hackathon.progress import track

started = time.time()
splitter = np.random.default_rng(0)
graphs, targets = [], []
val_graphs, val_targets = [], []
for suite in PUBLIC_SUITES:
    labels = load_labels(PARAMS_ROOT, suite)
    ids = np.asarray(local_split(suite)["train"][:N_TRAIN_PER_SUITE])
    # Split by SIMULATION before augmenting. Splitting the graph list instead would put
    # copies of the same simulation on both sides, and the validation loss would then be
    # measuring memorization rather than generalization.
    order = splitter.permutation(len(ids))
    n_val = int(round(VAL_FRACTION * len(ids)))
    val_ids, train_ids = ids[order[:n_val]], ids[order[n_val:]]

    for sim_id in track(train_ids, f"{suite}: reading catalogs and building graphs"):
        cat = load(suite, int(sim_id))
        for _ in range(N_AUG):
            cut = sample_mstar_threshold(splitter)
            graphs.append(catalog_to_graph(cat, mstar_min=cut, r_link=R_LINK_TUNED,
                                           use_mstar=USE_MSTAR))
            targets.append(labels[int(sim_id), :2])
    for sim_id in val_ids:               # fixed cut, no augmentation, like the test set
        val_graphs.append(catalog_to_graph(load(suite, int(sim_id)), r_link=R_LINK_TUNED,
                                           use_mstar=USE_MSTAR))
        val_targets.append(labels[int(sim_id), :2])

targets = np.asarray(targets)
val_targets = np.asarray(val_targets)
print(f"{len(graphs)} training graphs from {len(graphs) // N_AUG} simulations, "
      f"{len(val_graphs)} validation simulations, in {time.time() - started:.0f}s")

# Standardize the node and global features using the training set only.
x_all = np.concatenate([g["x"] for g in graphs])
u_all = np.stack([g["u"] for g in graphs])
STATS = {"x_mean": x_all.mean(0), "x_std": x_all.std(0) + 1e-8,
         "u_mean": u_all.mean(0), "u_std": u_all.std(0) + 1e-8}


def standardize(g):
    return {**g,
            "x": ((g["x"] - STATS["x_mean"]) / STATS["x_std"]).astype(np.float32),
            "u": ((g["u"] - STATS["u_mean"]) / STATS["u_std"]).astype(np.float32)}


graphs = [standardize(g) for g in graphs]
val_graphs = [standardize(g) for g in val_graphs]
y = torch.as_tensor((targets - PRIOR_LO) / (PRIOR_HI - PRIOR_LO),
                    dtype=torch.float32, device=DEVICE)
y_val = torch.as_tensor((val_targets - PRIOR_LO) / (PRIOR_HI - PRIOR_LO),
                        dtype=torch.float32, device=DEVICE)

In [ ]:
def forward(model, graph_list, index):
    batch = collate([graph_list[i] for i in index])
    return model(batch["x"].to(DEVICE), batch["edge_index"].to(DEVICE),
                 batch["edge_attr"].to(DEVICE), batch["u"].to(DEVICE),
                 batch["batch"].to(DEVICE))


rng = np.random.default_rng(0)
val_idx = np.arange(len(val_graphs))

torch.manual_seed(0)
model = DeSantiGNN(node_features=NODE_DIM, edge_features=3, n_global=1,
                   hidden=HIDDEN, n_layers=N_LAYERS, n_params=2).to(DEVICE)
# de Santi's recipe: Adam with a triangular cyclic learning rate.
optimizer = torch.optim.Adam(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
steps_per_epoch = int(np.ceil(len(graphs) / BATCH_SIZE))
schedule = torch.optim.lr_scheduler.CyclicLR(
    optimizer, base_lr=BASE_LR, max_lr=MAX_LR,
    step_size_up=int(round(EPOCHS / 4.8 * steps_per_epoch)), cycle_momentum=False)

history, best = [], (float("inf"), None)
started = time.time()
epochs = track(range(EPOCHS), "training")
for epoch in epochs:
    model.train()
    order = rng.permutation(len(graphs))
    running = 0.0
    for start in range(0, len(order), BATCH_SIZE):
        rows = order[start:start + BATCH_SIZE]
        optimizer.zero_grad(set_to_none=True)
        loss, _ = moment_loss(forward(model, graphs, rows), y[rows])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        schedule.step()
        running += float(loss.item()) * len(rows)
    model.eval()
    with torch.no_grad():
        val = float(moment_loss(forward(model, val_graphs, val_idx), y_val)[0].item())
    history.append((running / len(graphs), val))
    if val < best[0]:
        best = (val, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
    # The bar carries the numbers, so a long run is legible at a glance: `train` falling
    # while `val` climbs is overfitting, and `best` tells you which epoch will be kept.
    epochs.set_postfix(train=history[-1][0], val=val, best=best[0],
                       lr=schedule.get_last_lr()[0])

model.load_state_dict(best[1])
print(f"\nrestored the best-validation checkpoint (val = {best[0]:+.3f})")

In [ ]:
history = np.asarray(history)
fig, ax = plt.subplots(figsize=(6.4, 4))
ax.plot(history[:, 0], label="train")
ax.plot(history[:, 1], label="validation")
ax.axvline(int(np.argmin(history[:, 1])), ls="--", c="0.4", lw=1,
           label="checkpoint taken here")
ax.set_xlabel("epoch"); ax.set_ylabel("moment loss"); ax.legend()
plt.tight_layout()

## 6. Score it

On the held-out simulations, which the model has never seen. $R^2$ per target, never averaged
together.

In [ ]:
from kaai_hackathon.scoring import r2

N_TEST_PER_SUITE = 60


@torch.no_grad()
def predict(model, graph_list, batch_size=64):
    model.eval()
    out = []
    for start in range(0, len(graph_list), batch_size):
        index = np.arange(start, min(start + batch_size, len(graph_list)))
        mu, _ = DeSantiGNN.split_output(forward(model, graph_list, index))
        out.append(mu.cpu().numpy())
    return np.concatenate(out) * (PRIOR_HI - PRIOR_LO) + PRIOR_LO


test_graphs, test_truth, test_suite = [], [], []
for suite in PUBLIC_SUITES:
    labels = load_labels(PARAMS_ROOT, suite)
    for sim_id in track(local_split(suite)["test"][:N_TEST_PER_SUITE],
                        f"{suite}: scoring"):
        test_graphs.append(standardize(catalog_to_graph(load(suite, sim_id),
                                                        r_link=R_LINK_TUNED,
                                                        use_mstar=USE_MSTAR)))
        test_truth.append(labels[sim_id, :2])
        test_suite.append(suite)

test_truth = np.asarray(test_truth)
test_suite = np.asarray(test_suite)
prediction = predict(model, test_graphs)

print(f"{'':14s} {'Omega_m':>9s} {'sigma_8':>9s}")
for suite in PUBLIC_SUITES:
    m = test_suite == suite
    print(f"{suite:14s} " + " ".join(
        f"{r2(test_truth[m, j], prediction[m, j]):9.3f}" for j in range(2)))
print(f"{'macro':14s} " + " ".join(
    f"{np.mean([r2(test_truth[test_suite == s, j], prediction[test_suite == s, j]) for s in PUBLIC_SUITES]):9.3f}"
    for j in range(2)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for j, (ax, name, lo, hi) in enumerate(
        zip(axes, ("Omega_m", "sigma_8"), PRIOR_LO, PRIOR_HI)):
    for suite in PUBLIC_SUITES:
        m = test_suite == suite
        ax.scatter(test_truth[m, j], prediction[m, j], s=16, alpha=0.75, label=suite)
    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_xlabel(f"true {name}"); ax.set_ylabel(f"predicted {name}")
    ax.set_title(f"{name}   R^2 = {r2(test_truth[:, j], prediction[:, j]):.3f}")
axes[0].legend()
plt.tight_layout()

The $\sigma_8$ panel is the one to look at. Expect the points to sit in a horizontal band —
the model hedging towards the prior mean — while $\Omega_m$ tracks the diagonal. That gap is
the open research question this event is built around, and it is not a defect of this notebook.

## 7. Do the symmetries cost the trained model anything?

Section 2 showed the *features* are invariant. This runs the whole trained pipeline through
every condition, which is the thing that actually gets scored.

In [ ]:
from kaai_hackathon.conditions import (
    PUBLISHED_CONDITIONS, apply_condition, condition_seed,
)
from kaai_hackathon.catalog_io import read_catalog as _read

GROUP_FIELDS = ["GroupPos", "GroupCM", "GroupVel", "GroupMass",
                "GroupNsubs", "GroupFirstSub"]
SUBHALO_FIELDS = ["SubhaloPos", "SubhaloCM", "SubhaloVel", "SubhaloSpin",
                  "SubhaloMass", "SubhaloMassType", "SubhaloGrNr", "SubhaloParent"]
BASE_SEED = 2026
N_SHIFT_TEST = 40

shifted_scores = {}
for spec in track(PUBLISHED_CONDITIONS, "conditions"):
    graphs_s, truth_s = [], []
    for suite in PUBLIC_SUITES:
        labels = load_labels(PARAMS_ROOT, suite)
        for sim_id in local_split(suite)["test"][:N_SHIFT_TEST]:
            full = _read(catalog_path(suite, sim_id), group_fields=GROUP_FIELDS,
                         subhalo_fields=SUBHALO_FIELDS)
            seed = condition_seed(BASE_SEED, spec.name, suite, sim_id)
            graphs_s.append(standardize(catalog_to_graph(apply_condition(full, spec, seed),
                                                         r_link=R_LINK_TUNED,
                                                         use_mstar=USE_MSTAR)))
            truth_s.append(labels[sim_id, :2])
    truth_s = np.asarray(truth_s)
    p = predict(model, graphs_s)
    shifted_scores[spec.name] = (spec.tier,
                                 r2(truth_s[:, 0], p[:, 0]),
                                 r2(truth_s[:, 1], p[:, 1]))

clean = shifted_scores["clean"]
print(f"{'condition':14s} {'tier':5s} {'Omega_m':>9s} {'d':>8s} {'sigma_8':>9s} {'d':>8s}")
for name, (tier, om, s8) in shifted_scores.items():
    print(f"{name:14s} {tier:5s} {om:9.3f} {om - clean[1]:+8.3f} "
          f"{s8:9.3f} {s8 - clean[2]:+8.3f}")

Before reading anything into that table, note how few catalogs it is built on: 40 per suite,
120 in total. At that size an $R^2$ carries a scatter of a few hundredths, so a symmetry row
reading $+0.02$ or $-0.03$ is telling you about the sample, not about the model. Section 8
prints the same table for the shipped model, measured on the full held-out set, and that is
the one to trust. Run this cell with a larger `N_SHIFT_TEST` and the symmetry rows tighten
towards zero.

What survives the noise here is the size ordering. `translate` and `permute` land on top of
`clean` to three decimal places, because the invariance is exact and no amount of sampling
noise can move a number that did not change. `pos_noise_hi` costs whole units of $R^2$.
`vel_noise` costs `Omega_m` a great deal and `sigma_8` nothing, which is a real statement
about which parameter the velocities carry.

The general rule: whatever a corruption costs you is a result about your model's robustness,
while anything a symmetry costs you is a bug you can fix. And one sanity check before you
believe a clean-looking table — if **every** row is identical, corruption rows included, your
model is predicting a constant and the zeros mean nothing at all.

## 8. The shipped baseline

Everything above was scaled down so this notebook finishes. The shipped model is **the same
architecture and the same settings**, trained on all 900 public simulations per suite for 300
epochs. Its weights are in this repository and it is a working submission you can run today:

```
baseline/gnn/
    predict.py      load_model(model_dir) / predict(model, catalog_path)
    model.pt        172 KB, 42372 parameters
```

The checkpoint carries its own configuration and its own scores, so you can read off what it
is instead of trusting this text.

In [ ]:
import json
import torch

BASELINE = Path("..") / "baseline" / "gnn"
checkpoint = torch.load(BASELINE / "model.pt", map_location="cpu", weights_only=False)

print("configuration the weights were trained under:")
for key, value in checkpoint["args"].items():
    print(f"    {key:16s} {value}")

provenance = checkpoint["provenance"]
print(f"\ntrained on: {' + '.join(provenance['trained_on'])}")
print(f"training:   {json.dumps(provenance['training'])}")
print(f"\n{'condition':16s} {'Omega_m':>9s} {'sigma_8':>9s}")
for name, macro in provenance["scores"].items():
    print(f"{name:16s} {macro['Omega_m']:9.3f} {macro['sigma_8']:9.3f}")

Copy that directory, edit `predict.py`, and you have your own submission. Notebook 05 loads
this model, runs it, and walks through packaging one of your own.

## 9. Where the headroom is

Concrete things this baseline does not do:

1. **Two node features.** $v_z$ and $\log_{10}(1+M_\star)$, and nothing else. There are 47
   subhalo columns guaranteed to exist in every suite, the unseen one included: half-mass
   radius, velocity dispersion, gas fraction, black hole mass. All free to add, and all worth
   thinking about for whether they mean the same thing in a different simulation code.
2. **One linking length, tuned for one parameter.** 0.0384 came from a search over
   $\Omega_m$ alone. The same search run against $\sigma_8$ picks **0.0202** — half the
   scale — along with a deeper and wider network. The two parameters want different graphs,
   and this baseline predicts both from one. Two models, or one model reading two graph
   scales, is an obvious thing to try that nobody here has tried.
4. **Nothing about the shift.** The model never sees a shifted catalog during training. Shift
   augmentation is the obvious first thing to try, and the symmetry conditions tell you which
   ones you get for free from the architecture instead.
5. **Trained on all three suites at once.** Whether that helps or hurts on an unseen fourth
   code is an empirical question you can answer with a leave-one-suite-out split of your own —
   the same shape as the scored OOD condition.